In [ ]:

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import json

In [ ]:
OWM_API_KEY = "881e2360135b771a1e1979855f805eab"


In [ ]:
# Egyptian cities with coordinates
EGYPT_CITIES = {
    'Cairo': {'lat': 30.0444, 'lon': 31.2357, 'region': 'Greater Cairo'},
    'Alexandria': {'lat': 31.2001, 'lon': 29.9187, 'region': 'Mediterranean'},
    'Giza': {'lat': 30.0131, 'lon': 31.2089, 'region': 'Greater Cairo'},
    'Shubra El Kheima': {'lat': 30.1286, 'lon': 31.2422, 'region': 'Greater Cairo'},
    'Port Said': {'lat': 31.2653, 'lon': 32.3019, 'region': 'Canal'},
    'Suez': {'lat': 29.9668, 'lon': 32.5498, 'region': 'Canal'},
    'Luxor': {'lat': 25.6872, 'lon': 32.6396, 'region': 'Upper Egypt'},
    'Mansoura': {'lat': 31.0409, 'lon': 31.3785, 'region': 'Delta'},
    'Tanta': {'lat': 30.7865, 'lon': 31.0004, 'region': 'Delta'},
    'Asyut': {'lat': 27.1809, 'lon': 31.1837, 'region': 'Upper Egypt'},
    'Ismailia': {'lat': 30.5903, 'lon': 32.2722, 'region': 'Canal'},
    'Fayyum': {'lat': 29.3084, 'lon': 30.8405, 'region': 'Fayyum'},
    'Zagazig': {'lat': 30.5877, 'lon': 31.5022, 'region': 'Delta'},
    'Aswan': {'lat': 24.0889, 'lon': 32.8998, 'region': 'Upper Egypt'},
    'Damietta': {'lat': 31.4175, 'lon': 31.8144, 'region': 'Delta'},
}


In [ ]:
def get_nasa_historical_data(start_year=2022, end_year=2024):
    """
    Get NASA POWER data for Egyptian cities
    Parameters measured:
    - AOD_55: Aerosol Optical Depth (air quality indicator)
    - T2M: Temperature at 2 meters
    - PRECTOTCORR: Precipitation
    - RH2M: Relative Humidity

    FREE - No API key needed!
    """

    print("\n" + "=" * 70)
    print("🛰️  PART 1: NASA POWER - Historical Data Collection")
    print("=" * 70)
    print(f"📅 Period: {start_year} - {end_year}")
    print(f"📍 Cities: {len(EGYPT_CITIES)}")
    print()

    base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    all_data = []

    for city, coords in EGYPT_CITIES.items():

        params = {
            'parameters': 'AOD_55,T2M,PRECTOTCORR,RH2M,WS2M',
            # AOD_55 = Aerosol Optical Depth (pollution indicator)
            # T2M = Temperature
            # PRECTOTCORR = Precipitation
            # RH2M = Humidity
            # WS2M = Wind Speed
            'community': 'RE',
            'longitude': coords['lon'],
            'latitude': coords['lat'],
            'start': f"{start_year}0101",
            'end': f"{end_year}1231",
            'format': 'JSON'
        }

        try:
            print(f"📡 Fetching {city}...", end=" ")
            response = requests.get(base_url, params=params, timeout=30)

            if response.status_code == 200:
                data = response.json()

                if 'properties' in data and 'parameter' in data['properties']:
                    params_data = data['properties']['parameter']

                    # Get all dates
                    dates = list(params_data.get('AOD_55', {}).keys())

                    for date_str in dates:
                        try:
                            date_obj = datetime.strptime(date_str, '%Y%m%d')

                            # Get values (handle missing data)
                            aod = params_data['AOD_55'].get(date_str, -999)
                            temp = params_data['T2M'].get(date_str, -999)
                            precip = params_data['PRECTOTCORR'].get(date_str, -999)
                            humidity = params_data['RH2M'].get(date_str, -999)
                            wind = params_data['WS2M'].get(date_str, -999)

                            # Skip if AOD is missing (main indicator)
                            if aod == -999 or aod < 0:
                                continue

                            record = {
                                'city': city,
                                'region': coords['region'],
                                'latitude': coords['lat'],
                                'longitude': coords['lon'],
                                'date': date_obj,
                                'year': date_obj.year,
                                'month': date_obj.month,
                                'day': date_obj.day,
                                'day_of_week': date_obj.strftime('%A'),
                                'aod_55': aod,  # Main pollution indicator
                                'temperature_c': temp,
                                'precipitation_mm': precip if precip != -999 else 0,
                                'humidity_percent': humidity if humidity != -999 else None,
                                'wind_speed_ms': wind if wind != -999 else None,
                                'data_source': 'NASA POWER',
                                'data_type': 'Satellite Observation'
                            }

                            all_data.append(record)

                        except Exception as e:
                            continue

                    print(f"✅ {len(dates)} days")
                else:
                    print("⚠️  No data")
            else:
                print(f"❌ Error {response.status_code}")

            time.sleep(0.5)  # Be nice to NASA servers

        except Exception as e:
            print(f"❌ Error: {e}")

    # Create DataFrame
    df = pd.DataFrame(all_data)

    if not df.empty:
        # Convert AOD to Air Quality Categories
        # AOD interpretation:
        # 0.0-0.1 = Excellent
        # 0.1-0.2 = Good
        # 0.2-0.3 = Fair
        # 0.3-0.5 = Moderate
        # 0.5-1.0 = Poor
        # >1.0 = Very Poor

        df['air_quality_category'] = pd.cut(
            df['aod_55'],
            bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0, float('inf')],
            labels=['Excellent', 'Good', 'Fair', 'Moderate', 'Poor', 'Very Poor']
        )

        # Calculate pollution score (0-100)
        df['pollution_score'] = df['aod_55'].clip(0, 1.0) * 100

        print(f"\n✅ NASA Data Collection Complete!")
        print(f"📊 Total records: {len(df):,}")
        print(f"📅 Date range: {df['date'].min().date()} to {df['date'].max().date()}")
        print(f"📍 Cities: {df['city'].nunique()}")

    else:
        print("\n⚠️  No data collected from NASA")

    return df


In [ ]:

def get_openweathermap_recent_data(days_back=30):
    """
    Get recent detailed air quality data from OpenWeatherMap
    """

    print("\n" + "=" * 70)
    print("🌐 PART 2: OpenWeatherMap - Recent Detailed Data")
    print("=" * 70)
    print(f"📅 Last {days_back} days")
    print()

    url = "http://api.openweathermap.org/data/2.5/air_pollution/history"

    end_time = datetime.now()
    start_time = end_time - timedelta(days=days_back)

    all_data = []

    for city, coords in EGYPT_CITIES.items():

        params = {
            'lat': coords['lat'],
            'lon': coords['lon'],
            'start': int(start_time.timestamp()),
            'end': int(end_time.timestamp()),
            'appid': OWM_API_KEY
        }

        try:
            print(f"📡 Fetching {city}...", end=" ")
            response = requests.get(url, params=params)

            if response.status_code == 200:
                data = response.json()

                for item in data.get('list', []):
                    timestamp = datetime.fromtimestamp(item['dt'])

                    record = {
                        'city': city,
                        'region': coords['region'],
                        'latitude': coords['lat'],
                        'longitude': coords['lon'],
                        'timestamp': timestamp,
                        'date': timestamp.date(),
                        'year': timestamp.year,
                        'month': timestamp.month,
                        'day': timestamp.day,
                        'hour': timestamp.hour,
                        'day_of_week': timestamp.strftime('%A'),
                        'aqi': item['main']['aqi'],
                        'co': item['components'].get('co', 0),
                        'no': item['components'].get('no', 0),
                        'no2': item['components'].get('no2', 0),
                        'o3': item['components'].get('o3', 0),
                        'so2': item['components'].get('so2', 0),
                        'pm2_5': item['components'].get('pm2_5', 0),
                        'pm10': item['components'].get('pm10', 0),
                        'nh3': item['components'].get('nh3', 0),
                        'data_source': 'OpenWeatherMap',
                        'data_type': 'Ground Station Measurement'
                    }

                    all_data.append(record)

                print(f"✅ {len(data.get('list', []))} records")

            elif response.status_code == 401:
                print(f"❌ Invalid API Key!")
                break
            else:
                print(f"❌ Error {response.status_code}")

            time.sleep(0.2)

        except Exception as e:
            print(f"❌ Error: {e}")

    df = pd.DataFrame(all_data)

    if not df.empty:
        # Add AQI categories
        aqi_map = {1: 'Good', 2: 'Fair', 3: 'Moderate', 4: 'Poor', 5: 'Very Poor'}
        df['aqi_category'] = df['aqi'].map(aqi_map)

        print(f"\n✅ OpenWeatherMap Data Collection Complete!")
        print(f"📊 Total records: {len(df):,}")
        print(f"📅 Date range: {df['date'].min()} to {df['date'].max()}")
        print(f"📍 Cities: {df['city'].nunique()}")
    else:
        print("\n⚠️  No data collected from OpenWeatherMap")

    return df

In [ ]:

def analyze_nasa_data(df):
    """
    Analyze NASA historical data
    """

    print("\n" + "=" * 70)
    print("📊 NASA DATA ANALYSIS (2022-2024)")
    print("=" * 70)

    print("\n🏆 Cities ranked by average pollution (AOD):")
    city_avg = df.groupby('city')['aod_55'].mean().sort_values(ascending=False)
    for i, (city, aod) in enumerate(city_avg.head(10).items(), 1):
        print(f"{i}. {city:20s} | AOD: {aod:.3f}")

    print("\n📈 Yearly trends:")
    yearly = df.groupby('year')['aod_55'].agg(['mean', 'std', 'count'])
    print(yearly)

    print("\n📅 Monthly patterns:")
    monthly = df.groupby('month')['aod_55'].mean().sort_values(ascending=False)
    print(monthly.head(12))

    print("\n🌍 Regional comparison:")
    regional = df.groupby('region')['aod_55'].mean().sort_values(ascending=False)
    print(regional)

    return df

In [ ]:

def analyze_owm_data(df):
    """
    Analyze OpenWeatherMap recent data
    """

    print("\n" + "=" * 70)
    print("📊 OPENWEATHERMAP ANALYSIS (Last 30 days)")
    print("=" * 70)

    print("\n🏆 Cities ranked by PM2.5:")
    city_pm = df.groupby('city')['pm2_5'].mean().sort_values(ascending=False)
    for i, (city, pm) in enumerate(city_pm.head(10).items(), 1):
        print(f"{i}. {city:20s} | PM2.5: {pm:.2f} μg/m³")

    print("\n⏰ Hourly patterns (average PM2.5):")
    hourly = df.groupby('hour')['pm2_5'].mean().sort_values(ascending=False)
    print(hourly.head(24))

    print("\n📅 Day of week patterns:")
    dow = df.groupby('day_of_week')['pm2_5'].mean().sort_values(ascending=False)
    print(dow)

    print("\n💨 AQI distribution:")
    aqi_dist = df['aqi_category'].value_counts()
    print(aqi_dist)

    return df


In [ ]:

def export_all_data(nasa_df, owm_df):
    """
    Export all data to CSV and Excel files
    """

    print("\n" + "=" * 70)
    print("💾 EXPORTING DATA")
    print("=" * 70)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Export NASA data
    if not nasa_df.empty:
        nasa_csv = f"egypt_air_quality_nasa_2022_2024_{timestamp}.csv"
        nasa_df.to_csv(nasa_csv, index=False, encoding='utf-8-sig')
        print(f"✅ NASA CSV: {nasa_csv}")

        nasa_excel = f"egypt_air_quality_nasa_2022_2024_{timestamp}.xlsx"
        nasa_df.to_excel(nasa_excel, index=False, sheet_name='NASA Historical')
        print(f"✅ NASA Excel: {nasa_excel}")

    # Export OpenWeatherMap data
    if not owm_df.empty:
        owm_csv = f"egypt_air_quality_recent_{timestamp}.csv"
        owm_df.to_csv(owm_csv, index=False, encoding='utf-8-sig')
        print(f"✅ OpenWeatherMap CSV: {owm_csv}")

        owm_excel = f"egypt_air_quality_recent_{timestamp}.xlsx"
        owm_df.to_excel(owm_excel, index=False, sheet_name='Recent Data')
        print(f"✅ OpenWeatherMap Excel: {owm_excel}")

    # Export combined file
    if not nasa_df.empty and not owm_df.empty:
        combined_excel = f"egypt_air_quality_complete_{timestamp}.xlsx"
        with pd.ExcelWriter(combined_excel, engine='openpyxl') as writer:
            nasa_df.to_excel(writer, sheet_name='NASA_2022-2024', index=False)
            owm_df.to_excel(writer, sheet_name='Recent_30days', index=False)
        print(f"✅ Combined Excel: {combined_excel}")

    print("\n🎉 All data exported successfully!")
    print("📁 Files are ready for Power BI!")


In [ ]:

def main():
    """
    Main execution function
    """

    print("\n" + "=" * 70)
    print("🇪🇬 EGYPT AIR QUALITY - COMPLETE DATA COLLECTION")
    print("=" * 70)
    print("🛰️  NASA POWER: Historical data (2022-2024)")
    print("🌐 OpenWeatherMap: Recent detailed data (last 30 days)")
    print()

    # Part 1: Collect NASA data (Historical)
    nasa_df = get_nasa_historical_data(start_year=2022, end_year=2024)

    # Part 2: Collect OpenWeatherMap data (Recent)
    owm_df = get_openweathermap_recent_data(days_back=30)

    # Part 3: Analysis
    if not nasa_df.empty:
        analyze_nasa_data(nasa_df)

    if not owm_df.empty:
        analyze_owm_data(owm_df)

    # Part 4: Export
    export_all_data(nasa_df, owm_df)

    print("\n" + "=" * 70)
    print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
    print("=" * 70)

    return nasa_df, owm_df


In [ ]:

if __name__ == "__main__":
    nasa_data, owm_data = main()

    print("\n📊 Data Summary:")
    if not nasa_data.empty:
        print(f"\n🛰️  NASA Data: {len(nasa_data):,} records")
        print(nasa_data.head())

    if not owm_data.empty:
        print(f"\n🌐 OpenWeatherMap Data: {len(owm_data):,} records")
        print(owm_data.head())


🇪🇬 EGYPT AIR QUALITY - COMPLETE DATA COLLECTION
🛰️  NASA POWER: Historical data (2022-2024)
🌐 OpenWeatherMap: Recent detailed data (last 30 days)


🛰️  PART 1: NASA POWER - Historical Data Collection
📅 Period: 2022 - 2024
📍 Cities: 15

📡 Fetching Cairo... ✅ 1096 days
📡 Fetching Alexandria... ✅ 1096 days
📡 Fetching Giza... ✅ 1096 days
📡 Fetching Shubra El Kheima... ✅ 1096 days
📡 Fetching Port Said... ✅ 1096 days
📡 Fetching Suez... ✅ 1096 days
📡 Fetching Luxor... ✅ 1096 days
📡 Fetching Mansoura... ✅ 1096 days
📡 Fetching Tanta... ✅ 1096 days
📡 Fetching Asyut... ✅ 1096 days
📡 Fetching Ismailia... ✅ 1096 days
📡 Fetching Fayyum... ✅ 1096 days
📡 Fetching Zagazig... ✅ 1096 days
📡 Fetching Aswan... ✅ 1096 days
📡 Fetching Damietta... ✅ 1096 days

✅ NASA Data Collection Complete!
📊 Total records: 16,440
📅 Date range: 2022-01-01 to 2024-12-31
📍 Cities: 15

🌐 PART 2: OpenWeatherMap - Recent Detailed Data
📅 Last 30 days

📡 Fetching Cairo... ✅ 721 records
📡 Fetching Alexandria... ✅ 721 records
📡 Fet